# 13 — Silver — Sanções (CEIS + CNEP)

## Purpose
Build `silver_ceis` and `silver_cnep` from Portal da Transparência.
These tables feed the Gold sanctions facts and the serving layer.

## Input
| Source | Path | Bronze rows |
|---|---|---|
| CEIS | `data/bronze/transp_ceis/` | 22,498 |
| CNEP | `data/bronze/transp_cnep/` | 1,620 |

## Output
| Table | Path | Grain | Expected rows |
|---|---|---|---|
| `silver_ceis` | `data/silver/silver_ceis/data.parquet` | 1 row per sanction | ~13,562 PJ |
| `silver_cnep` | `data/silver/silver_cnep/data.parquet` | 1 row per sanction | ~1,580 PJ |

## Steps
1. Imports and configuration
2. Key findings from EDA
3. Build silver_ceis
4. Build silver_cnep
5. Validation

## Notes
- Filter: `pessoa_tipo IN ('Entidades Empresariais Privadas', 'Entidades Sem Fins Lucrativos')`
- `cnpj_normalized`: `REGEXP_REPLACE` removes all non-alphanumeric chars — ADR-002
- `cnpj_basico`: `LEFT(cnpj_normalized, 8)` — root for JOIN with `silver_identidade`
- Dates: `TRY_STRPTIME(val, '%d/%m/%Y')::DATE` | sentinel `'Sem informacao'` → NULL
- `valorMulta` (CNEP only): BR decimal → `DECIMAL(15,2)` | `'0,00'` → `0.00` (not NULL)
- `orgaoSancionador_siglaUf` NULL for federal organs — correct, not missing data
- No partitioning: ~15k rows total — small files antipattern outweighs benefit
- CEIS and CNEP kept separate: different legal bases, CNEP has `valorMulta`
- ADR-002: CNPJ always VARCHAR — never cast to INTEGER

## Step 1 — Imports and configuration

In [ ]:
# =============================================================================
# Step 1 — Imports and configuration
# =============================================================================

import sys
from pathlib import Path

# --- utils -------------------------------------------------------------------
# Resolve project root first so utils/ is importable regardless of how
# VS Code or Jupyter resolved the working directory.
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    _root_candidate = _nb_file.parent.parent
except NameError:
    _root_candidate = Path.cwd().parent

sys.path.insert(0, str(_root_candidate / "utils"))

from paths       import get_project_root, bronze_path, silver_path, ensure_dir, to_sql_path
from duckdb_utils import get_connection, scalar, cnpj_normalize_expr, safe_date_expr
from validation  import CheckSuite

from datetime import datetime, timezone

# --- Paths -------------------------------------------------------------------
ROOT     = get_project_root()
OUT_CEIS = ROOT / "data" / "silver" / "silver_ceis"
OUT_CNEP = ROOT / "data" / "silver" / "silver_cnep"

BD_CEIS  = bronze_path(ROOT, "transp_ceis",  glob=True)
BD_CNEP  = bronze_path(ROOT, "transp_cnep",  glob=True)
SD_CEIS  = silver_path(ROOT, "silver_ceis")   # .../silver_ceis/data.parquet
SD_CNEP  = silver_path(ROOT, "silver_cnep")   # .../silver_cnep/data.parquet

STARTED_AT = datetime.now(timezone.utc)

# Idempotent — always rebuild (tables are small, < 15k rows)
for out_dir in [OUT_CEIS, OUT_CNEP]:
    import shutil
    if out_dir.exists():
        shutil.rmtree(out_dir)
    ensure_dir(out_dir)

# --- Connection --------------------------------------------------------------
con = get_connection()   # in-memory, 4 threads, 8 GB

import duckdb
print(f"ROOT         : {ROOT}")
print(f"OUT_CEIS     : {OUT_CEIS}")
print(f"OUT_CNEP     : {OUT_CNEP}")
print(f"Started at   : {STARTED_AT.isoformat()}")
print(f"DuckDB ver.  : {duckdb.__version__}")
print()
print("Configuration ready.")

In [ ]:
# =============================================================================
# Step 2 — Key findings from EDA
# =============================================================================

# CEIS vs CNEP
# ┌─────────────────────────┬──────────────────────────────┬────────────────────────────┐
# │ Aspect                  │ CEIS                         │ CNEP                       │
# ├─────────────────────────┼──────────────────────────────┼────────────────────────────┤
# │ Legal basis             │ Lei 8.666, Lei 10.520, etc.  │ Lei 12.846/2013 (Anti-Corr)│
# │ valorMulta              │ Does not exist               │ Exists — BR decimal format │
# │ Bronze rows             │ 22,498                       │ 1,620                      │
# │ PJ after filter         │ 13,562 (60.3%)               │ 1,580 (97.5%)              │
# │ PJ without CNPJ         │ 0                            │ 0                          │
# │ PJ without organ UF     │ 743 (5.5%)                   │ 642 (40.6%)                │
# └─────────────────────────┴──────────────────────────────┴────────────────────────────┘
#
# orgaoSancionador_siglaUf NULL — expected, not missing data
#   Federal organs (Receita Federal, Petrobras, BB, Ministries, Armed Forces)
#   operate nationally and have no UF. Silver keeps NULL — do not impute.
#
# Date sentinel: 'Sem informacao' (not NULL, not empty string)
#   Appears in: dataFimSancao, dataPublicacaoSancao, dataTransitadoJulgado,
#               dataOrigemInformacao
#
# valorMulta (CNEP only)
#   Brazilian decimal: dot = thousands sep, comma = decimal sep
#   '390.630,47' -> 390630.47
#   705 records (43.5%) have '0,00' — kept as 0.00 (valid: no monetary penalty)

#print("Step 2 — EDA findings loaded (no execution needed).")

## Step 3 — Build silver_ceis

In [ ]:
# =============================================================================
# Step 3 — Build silver_ceis
# =============================================================================

# SQL expression helpers — generated once, reused in the COPY statement
_cnpj_expr  = cnpj_normalize_expr("pessoa_cnpjFormatado")

# safe_date_expr returns the CASE WHEN block for each date field
_dt_inicio  = safe_date_expr("dataInicioSancao",     "'Sem informacao'", "'%d/%m/%Y'")
_dt_fim     = safe_date_expr("dataFimSancao",         "'Sem informacao'", "'%d/%m/%Y'")
_dt_pub     = safe_date_expr("dataPublicacaoSancao",  "'Sem informacao'", "'%d/%m/%Y'")
_dt_trans   = safe_date_expr("dataTransitadoJulgado", "'Sem informacao'", "'%d/%m/%Y'")
_dt_origem  = safe_date_expr("dataOrigemInformacao",  "'Sem informacao'", "'%d/%m/%Y'")

t0 = datetime.now()

sql_ceis = f"""
    COPY (
        SELECT
            {_cnpj_expr}                                                    AS cnpj_normalized,
            LEFT({_cnpj_expr}, 8)                                           AS cnpj_basico,
            id                                                              AS sancao_id,
            tipoSancao_descricaoResumida                                    AS tipo_sancao,
            tipoSancao_descricaoPortal                                      AS tipo_sancao_detalhe,
            {_dt_inicio}                                                    AS data_inicio_sancao,
            {_dt_fim}                                                       AS data_fim_sancao,
            {_dt_pub}                                                       AS data_publicacao_sancao,
            {_dt_trans}                                                     AS data_transitado_julgado,
            {_dt_origem}                                                    AS data_origem_informacao,
            pessoa_nome                                                     AS sancionado_nome,
            pessoa_razaoSocialReceita                                       AS sancionado_razao_social,
            pessoa_nomeFantasiaReceita                                      AS sancionado_nome_fantasia,
            pessoa_tipo                                                     AS sancionado_tipo,
            orgaoSancionador_nome                                           AS orgao_sancionador_nome,
            orgaoSancionador_siglaUf                                        AS orgao_sancionador_uf,
            orgaoSancionador_esfera                                         AS orgao_sancionador_esfera,
            orgaoSancionador_poder                                          AS orgao_sancionador_poder,
            numeroProcesso                                                  AS numero_processo,
            fundamentacao,
            textoPublicacao                                                 AS texto_publicacao,
            linkPublicacao                                                  AS link_publicacao,
            informacoesAdicionaisDoOrgaoSancionador                         AS informacoes_adicionais,
            'ceis'                                                          AS _fonte,
            CURRENT_TIMESTAMP                                               AS _silver_loaded_at
        FROM read_parquet('{BD_CEIS}')
        WHERE pessoa_tipo IN (
            'Entidades Empresariais Privadas',
            'Entidades Sem Fins Lucrativos'
        )
        -- Deduplication: same sancao_id published multiple times with different
        -- fundamentacao field order — source data quality issue (Portal da Transparência)
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY id
            ORDER BY CURRENT_TIMESTAMP
        ) = 1
    )
    TO '{SD_CEIS}'
    (FORMAT PARQUET)
"""

con.execute(sql_ceis)

elapsed = (datetime.now() - t0).total_seconds()
n       = scalar(con, f"SELECT count(*) FROM read_parquet('{SD_CEIS}')")
size_mb = (OUT_CEIS / "data.parquet").stat().st_size / (1024 * 1024)

print(f"silver_ceis built in {elapsed:.1f}s")
print(f"Rows   : {n:,}")
print(f"Size   : {size_mb:.1f} MB")

## Step 4 — Build silver_cnep

In [ ]:
# =============================================================================
# Step 4 — Build silver_cnep
# =============================================================================

# Same date expressions as CEIS — reused from Step 3 variables
# Additional field: valorMulta (BR decimal -> DECIMAL(15,2))
#   REPLACE('.', '') removes thousands separator
#   REPLACE(',', '.') converts decimal separator
#   TRY_CAST returns NULL on unexpected values instead of raising

t0 = datetime.now()

sql_cnep = f"""
    COPY (
        SELECT
            {_cnpj_expr}                                                    AS cnpj_normalized,
            LEFT({_cnpj_expr}, 8)                                           AS cnpj_basico,
            id                                                              AS sancao_id,
            tipoSancao_descricaoResumida                                    AS tipo_sancao,
            tipoSancao_descricaoPortal                                      AS tipo_sancao_detalhe,
            {_dt_inicio}                                                    AS data_inicio_sancao,
            {_dt_fim}                                                       AS data_fim_sancao,
            {_dt_pub}                                                       AS data_publicacao_sancao,
            {_dt_trans}                                                     AS data_transitado_julgado,
            {_dt_origem}                                                    AS data_origem_informacao,
            TRY_CAST(
                REPLACE(REPLACE(valorMulta, '.', ''), ',', '.')
                AS DECIMAL(15, 2)
            )                                                               AS valor_multa,
            pessoa_nome                                                     AS sancionado_nome,
            pessoa_razaoSocialReceita                                       AS sancionado_razao_social,
            pessoa_nomeFantasiaReceita                                      AS sancionado_nome_fantasia,
            pessoa_tipo                                                     AS sancionado_tipo,
            orgaoSancionador_nome                                           AS orgao_sancionador_nome,
            orgaoSancionador_siglaUf                                        AS orgao_sancionador_uf,
            orgaoSancionador_esfera                                         AS orgao_sancionador_esfera,
            orgaoSancionador_poder                                          AS orgao_sancionador_poder,
            numeroProcesso                                                  AS numero_processo,
            fundamentacao,
            textoPublicacao                                                 AS texto_publicacao,
            linkPublicacao                                                  AS link_publicacao,
            informacoesAdicionaisDoOrgaoSancionador                         AS informacoes_adicionais,
            'cnep'                                                          AS _fonte,
            CURRENT_TIMESTAMP                                               AS _silver_loaded_at
        FROM read_parquet('{BD_CNEP}')
        WHERE pessoa_tipo IN (
            'Entidades Empresariais Privadas',
            'Entidades Sem Fins Lucrativos'
        )
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY id
            ORDER BY CURRENT_TIMESTAMP
        ) = 1
    )
    TO '{SD_CNEP}'
    (FORMAT PARQUET)
"""

con.execute(sql_cnep)
con.close()

elapsed = (datetime.now() - t0).total_seconds()
n       = scalar(get_connection(), f"SELECT count(*) FROM read_parquet('{SD_CNEP}')")
size_mb = (OUT_CNEP / "data.parquet").stat().st_size / (1024 * 1024)

print(f"silver_cnep built in {elapsed:.1f}s")
print(f"Rows   : {n:,}")
print(f"Size   : {size_mb:.1f} MB")

## Step 5 — Validation

In [ ]:
# =============================================================================
# Step 5 — Validation
# =============================================================================

con_val = get_connection()

# Calcula o esperado diretamente do Bronze com o mesmo filtro da transformação
# CEIS e CNEP: inclui deduplicação por id — espelha o QUALIFY dos Steps 3 e 4
PJ_FILTER = """
    pessoa_tipo IN (
        'Entidades Empresariais Privadas',
        'Entidades Sem Fins Lucrativos'
    )
"""

EXPECTED_CEIS = scalar(con_val, f"""
    SELECT COUNT(*) FROM (
        SELECT id
        FROM read_parquet('{BD_CEIS}')
        WHERE {PJ_FILTER}
        QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY CURRENT_TIMESTAMP) = 1
    )
""")

EXPECTED_CNEP = scalar(con_val, f"""
    SELECT COUNT(*) FROM (
        SELECT id
        FROM read_parquet('{BD_CNEP}')
        WHERE {PJ_FILTER}
        QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY CURRENT_TIMESTAMP) = 1
    )
""")

print(f"Expected CEIS (PJ scope, deduped) : {EXPECTED_CEIS:,}")
print(f"Expected CNEP (PJ scope, deduped) : {EXPECTED_CNEP:,}")

for src, sd, out_dir, expected in [
    ("silver_ceis", SD_CEIS, OUT_CEIS, EXPECTED_CEIS),
    ("silver_cnep", SD_CNEP, OUT_CNEP, EXPECTED_CNEP),
]:
    suite = CheckSuite(src)
    n     = scalar(con_val, f"SELECT count(*) FROM read_parquet('{sd}')")

    # 1. Row count
    suite.add("Row count matches expected PJ scope", n == expected, f"{n:,} rows (expected {expected:,})")

    # 2. cnpj_normalized integrity
    nulls     = scalar(con_val, f"SELECT count(*) FROM read_parquet('{sd}') WHERE cnpj_normalized IS NULL")
    wrong_len = scalar(con_val, f"SELECT count(*) FROM read_parquet('{sd}') WHERE length(cnpj_normalized) != 14")
    has_punct = scalar(con_val, f"""
        SELECT count(*) FROM read_parquet('{sd}')
        WHERE cnpj_normalized LIKE '%.%'
           OR cnpj_normalized LIKE '%/%'
           OR cnpj_normalized LIKE '%-%'
    """)
    suite.add("cnpj_normalized — no nulls",       nulls     == 0, f"{nulls} nulls")
    suite.add("cnpj_normalized — length = 14",    wrong_len == 0, f"{wrong_len} wrong length")
    suite.add("cnpj_normalized — no punctuation", has_punct == 0, f"{has_punct} with punctuation")

    # 3. Critical fields never null
    for col in ["cnpj_basico", "sancao_id", "tipo_sancao", "data_inicio_sancao"]:
        n_null = scalar(con_val, f"SELECT count(*) FROM read_parquet('{sd}') WHERE {col} IS NULL")
        suite.add(f"{col} — no nulls", n_null == 0, f"{n_null} nulls")

    # 4. Unicidade de sancao_id por cnpj (fonte com duplicatas conhecidas — Portal da Transparência)
    if src in ("silver_ceis", "silver_cnep"):
        dupes = scalar(con_val, f"""
            SELECT COUNT(*) FROM (
                SELECT cnpj_normalized, sancao_id, COUNT(*) AS n
                FROM read_parquet('{sd}')
                GROUP BY 1, 2
                HAVING n > 1
            )
        """)
        suite.add("sancao_id unique per cnpj", dupes == 0,
                  f"{dupes} duplicate sancao_id groups")

    # 5. valorMulta (CNEP only)
    if src == "silver_cnep":
        n_null_multa = scalar(con_val, f"SELECT count(*) FROM read_parquet('{sd}') WHERE valor_multa IS NULL")
        max_multa    = scalar(con_val, f"SELECT max(valor_multa) FROM read_parquet('{sd}')")
        suite.add("valor_multa — no nulls",   n_null_multa == 0, f"{n_null_multa} nulls")
        suite.add("valor_multa — max sanity", max_multa > 0,     f"max = {max_multa:,.2f}")

    suite.report()
    suite.assert_all_pass()

    # 6. Sample rows
    print(f"\n  Sample rows — {src}:")
    sample = con_val.execute(f"""
        SELECT cnpj_normalized, sancionado_razao_social,
               tipo_sancao, data_inicio_sancao, data_fim_sancao,
               orgao_sancionador_nome, orgao_sancionador_uf
        FROM read_parquet('{sd}')
        LIMIT 4
    """).fetchall()
    for row in sample:
        def fmt(v): return (str(v) if v is not None else "NULL")[:22]
        print("    " + "  ".join(fmt(v) for v in row))
    print()

con_val.close()

total_elapsed = (datetime.now(timezone.utc) - STARTED_AT).total_seconds()
print(f"Notebook completed in {total_elapsed:.0f}s ({total_elapsed/60:.1f} min)")